In [71]:
# ============================================================
# UNCERTAINTY CATEGORIZATION
# ============================================================

# Calculate percentile thresholds
uncertainty_p33 = np.percentile(
    mc_variance,
    33
)

uncertainty_p67 = np.percentile(
    mc_variance,
    67
)

# ------------------------------------------------------------
# Assign uncertainty categories
# ------------------------------------------------------------

uncertainty_levels = np.where(
    mc_variance <= uncertainty_p33,
    "Low",
    np.where(
        mc_variance <= uncertainty_p67,
        "Medium",
        "High"
    )
)

# ============================================================
# DISPLAY THRESHOLDS
# ============================================================

print("UNCERTAINTY CATEGORIZATION")
print("=" * 60)

print(
    f"33rd percentile threshold: "
    f"{uncertainty_p33:.10f}"
)

print(
    f"67th percentile threshold: "
    f"{uncertainty_p67:.10f}"
)

# ============================================================
# COUNT EACH CATEGORY
# ============================================================

unique_levels, level_counts = np.unique(
    uncertainty_levels,
    return_counts=True
)

print("\nUNCERTAINTY LEVEL DISTRIBUTION")
print("-" * 60)

for level, count in zip(
    unique_levels,
    level_counts
):

    percentage = (
        count / len(uncertainty_levels)
    ) * 100

    print(
        f"{level:8s}: "
        f"{count:6d} patients "
        f"({percentage:.2f}%)"
    )

# ============================================================
# VERIFY ALL PATIENTS WERE ASSIGNED
# ============================================================

print("\nSANITY CHECK")
print("-" * 60)

print(
    "Total patients:",
    len(uncertainty_levels)
)

print(
    "Expected patients:",
    len(test_true_labels)
)

print(
    "Missing uncertainty levels:",
    np.sum(
        pd.isna(uncertainty_levels)
    )
)

print(
    "Unique uncertainty levels:",
    np.unique(
        uncertainty_levels
    )
)

# ============================================================
# FIRST 10 PATIENTS
# ============================================================

print("\nFIRST 10 PATIENTS")
print("-" * 60)

for i in range(10):

    print(
        f"Patient {i+1:02d} | "
        f"Mean = {mc_mean[i]:.6f} | "
        f"Variance = {mc_variance[i]:.10f} | "
        f"Uncertainty = {uncertainty_levels[i]} | "
        f"True label = {int(test_true_labels[i])}"
    )

UNCERTAINTY CATEGORIZATION
33rd percentile threshold: 0.0010119812
67th percentile threshold: 0.0018078083

UNCERTAINTY LEVEL DISTRIBUTION
------------------------------------------------------------
High    :  12557 patients (33.00%)
Low     :  12557 patients (33.00%)
Medium  :  12938 patients (34.00%)

SANITY CHECK
------------------------------------------------------------
Total patients: 38052
Expected patients: 38052
Missing uncertainty levels: 0
Unique uncertainty levels: ['High' 'Low' 'Medium']

FIRST 10 PATIENTS
------------------------------------------------------------
Patient 01 | Mean = 0.546721 | Variance = 0.0019651442 | Uncertainty = High | True label = 1
Patient 02 | Mean = 0.113666 | Variance = 0.0007691144 | Uncertainty = Low | True label = 0
Patient 03 | Mean = 0.525235 | Variance = 0.0039195600 | Uncertainty = High | True label = 0
Patient 04 | Mean = 0.512595 | Variance = 0.0022348210 | Uncertainty = High | True label = 0
Patient 05 | Mean = 0.256157 | Variance =

In [72]:
# ============================================================
# UNCERTAINTY GROUP VALIDATION
# ============================================================

print("UNCERTAINTY GROUP VALIDATION")
print("=" * 70)

for level in ["Low", "Medium", "High"]:

    # Select patients belonging to this uncertainty level
    mask = uncertainty_levels == level

    # Extract their variances
    group_variance = mc_variance[mask]

    # Extract their standard deviations
    group_std = mc_std[mask]

    # Extract their mean predictions
    group_mean = mc_mean[mask]

    print(f"\n{level.upper()} UNCERTAINTY")
    print("-" * 70)

    print(f"Number of patients : {len(group_variance):,}")

    print(
        f"Mean variance      : "
        f"{np.mean(group_variance):.10f}"
    )

    print(
        f"Median variance    : "
        f"{np.median(group_variance):.10f}"
    )

    print(
        f"Mean std deviation : "
        f"{np.mean(group_std):.6f}"
    )

    print(
        f"Mean prediction    : "
        f"{np.mean(group_mean):.6f}"
    )

# ============================================================
# ORDERING CHECK
# ============================================================

low_mean_variance = np.mean(
    mc_variance[uncertainty_levels == "Low"]
)

medium_mean_variance = np.mean(
    mc_variance[uncertainty_levels == "Medium"]
)

high_mean_variance = np.mean(
    mc_variance[uncertainty_levels == "High"]
)

print("\n" + "=" * 70)
print("ORDERING CHECK")
print("=" * 70)

print(
    f"Low    mean variance    : "
    f"{low_mean_variance:.10f}"
)

print(
    f"Medium mean variance    : "
    f"{medium_mean_variance:.10f}"
)

print(
    f"High   mean variance    : "
    f"{high_mean_variance:.10f}"
)

print("\nExpected relationship:")
print("Low < Medium < High")

if (
    low_mean_variance
    < medium_mean_variance
    < high_mean_variance
):
    print("\n✓ UNCERTAINTY ORDERING VERIFIED")
else:
    print("\n✗ WARNING: UNCERTAINTY ORDERING NOT VERIFIED")

UNCERTAINTY GROUP VALIDATION

LOW UNCERTAINTY
----------------------------------------------------------------------
Number of patients : 12,557
Mean variance      : 0.0004525014
Median variance    : 0.0004086724
Mean std deviation : 0.019519
Mean prediction    : 0.080018

MEDIUM UNCERTAINTY
----------------------------------------------------------------------
Number of patients : 12,938
Mean variance      : 0.0014061445
Median variance    : 0.0014068999
Mean std deviation : 0.037380
Mean prediction    : 0.330575

HIGH UNCERTAINTY
----------------------------------------------------------------------
Number of patients : 12,557
Mean variance      : 0.0026478516
Median variance    : 0.0023815674
Mean std deviation : 0.050850
Mean prediction    : 0.475580

ORDERING CHECK
Low    mean variance    : 0.0004525014
Medium mean variance    : 0.0014061445
High   mean variance    : 0.0026478516

Expected relationship:
Low < Medium < High

✓ UNCERTAINTY ORDERING VERIFIED


In [77]:
# ============================================================
# STEP 1A — Uncertainty Stratification
# ============================================================

# PRD-defined uncertainty thresholds
LOW_THRESHOLD = 0.05
HIGH_THRESHOLD = 0.15

# Assign each test instance to an uncertainty stratum
uncertainty_stratum = np.where(
    mc_variance < LOW_THRESHOLD,
    "LOW",
    np.where(
        mc_variance < HIGH_THRESHOLD,
        "MEDIUM",
        "HIGH"
    )
)

# Count instances in each stratum
stratum_counts = pd.Series(uncertainty_stratum).value_counts()

# Ensure consistent ordering
stratum_counts = stratum_counts.reindex(
    ["LOW", "MEDIUM", "HIGH"],
    fill_value=0
)

print("PRD Uncertainty Stratification")
print("=" * 40)
print(f"LOW    (σ² < 0.05):       {stratum_counts['LOW']:,}")
print(f"MEDIUM (0.05 ≤ σ² < 0.15): {stratum_counts['MEDIUM']:,}")
print(f"HIGH   (σ² ≥ 0.15):       {stratum_counts['HIGH']:,}")
print("=" * 40)
print(f"Total:                    {stratum_counts.sum():,}")

PRD Uncertainty Stratification
LOW    (σ² < 0.05):       38,052
MEDIUM (0.05 ≤ σ² < 0.15): 0
HIGH   (σ² ≥ 0.15):       0
Total:                    38,052


In [81]:
# ============================================================
# STEP 1B — Adjusted Uncertainty Threshold Check
# ============================================================

ADJUSTED_LOW_THRESHOLD = 0.03
ADJUSTED_HIGH_THRESHOLD = 0.10

adjusted_stratum = np.where(
    mc_variance < ADJUSTED_LOW_THRESHOLD,
    "LOW",
    np.where(
        mc_variance < ADJUSTED_HIGH_THRESHOLD,
        "MEDIUM",
        "HIGH"
    )
)

adjusted_counts = pd.Series(adjusted_stratum).value_counts()

adjusted_counts = adjusted_counts.reindex(
    ["LOW", "MEDIUM", "HIGH"],
    fill_value=0
)

print("Adjusted PRD Uncertainty Stratification")
print("=" * 45)
print(f"LOW    (σ² < 0.03):          {adjusted_counts['LOW']:,}")
print(f"MEDIUM (0.03 ≤ σ² < 0.10): {adjusted_counts['MEDIUM']:,}")
print(f"HIGH   (σ² ≥ 0.10):         {adjusted_counts['HIGH']:,}")
print("=" * 45)
print(f"Total:                       {adjusted_counts.sum():,}")

print("\nMinimum stratum requirement: 150 instances")
print(f"LOW ≥ 150:    {adjusted_counts['LOW'] >= 150}")
print(f"MEDIUM ≥ 150: {adjusted_counts['MEDIUM'] >= 150}")
print(f"HIGH ≥ 150:   {adjusted_counts['HIGH'] >= 150}")

Adjusted PRD Uncertainty Stratification
LOW    (σ² < 0.03):          38,052
MEDIUM (0.03 ≤ σ² < 0.10): 0
HIGH   (σ² ≥ 0.10):         0
Total:                       38,052

Minimum stratum requirement: 150 instances
LOW ≥ 150:    True
MEDIUM ≥ 150: False
HIGH ≥ 150:   False


In [82]:
# ============================================================
# STEP 1C — Full MC Dropout Variance Distribution
# ============================================================

variance_percentiles = {
    "50th (Median)": np.percentile(mc_variance, 50),
    "90th": np.percentile(mc_variance, 90),
    "95th": np.percentile(mc_variance, 95),
    "99th": np.percentile(mc_variance, 99),
    "99.9th": np.percentile(mc_variance, 99.9),
    "Maximum": np.max(mc_variance)
}

print("MC Dropout Predictive Variance Distribution")
print("=" * 50)

for percentile, value in variance_percentiles.items():
    print(f"{percentile:<18}: {value:.10f}")

print("=" * 50)

# Additional verification of the MC inference state
print("\nMC Dropout Inference Configuration")
print("=" * 50)

print(f"Number of MC passes (T): {mc_predictions.shape[0]}")
print(f"Number of test instances: {mc_predictions.shape[1]}")

print("\nCurrent model state:")
print(f"model.training = {model.training}")

print("\nDropout layer states:")
print(f"drop1.training = {model.drop1.training}")
print(f"drop2.training = {model.drop2.training}")
print(f"drop3.training = {model.drop3.training}")

print("\nBatchNorm layer states:")
print(f"bn1.training = {model.bn1.training}")
print(f"bn2.training = {model.bn2.training}")

MC Dropout Predictive Variance Distribution
50th (Median)     : 0.0014068999
90th              : 0.0027569428
95th              : 0.0033079339
99th              : 0.0048498134
99.9th            : 0.0085418466
Maximum           : 0.0145257190

MC Dropout Inference Configuration
Number of MC passes (T): 50
Number of test instances: 38052

Current model state:
model.training = False

Dropout layer states:
drop1.training = False
drop2.training = False
drop3.training = False

BatchNorm layer states:
bn1.training = False
bn2.training = False


In [85]:
# ============================================================
# STEP 1C — Correct MC Dropout Inference
# ============================================================

T = 50

# Activate training mode so Dropout layers are ACTIVE
model.train()

mc_predictions_corrected = []

with torch.no_grad():
    for _ in range(T):
        batch_predictions = []

        for X_batch, _ in test_loader:
            predictions = model(X_batch)
            batch_predictions.append(predictions.squeeze(1).cpu())

        pass_predictions = torch.cat(batch_predictions)
        mc_predictions_corrected.append(pass_predictions)

# Convert to tensor
mc_predictions_corrected = torch.stack(mc_predictions_corrected)

# Calculate mean prediction and predictive variance
mc_mean_corrected = mc_predictions_corrected.mean(dim=0).numpy()
mc_variance_corrected = mc_predictions_corrected.var(dim=0).numpy()
mc_std_corrected = np.sqrt(mc_variance_corrected)

print("Corrected MC Dropout Inference")
print("=" * 50)
print(f"MC passes (T): {T}")
print(f"Prediction shape: {mc_predictions_corrected.shape}")
print(f"Mean prediction shape: {mc_mean_corrected.shape}")
print(f"Variance shape: {mc_variance_corrected.shape}")
print("=" * 50)

# ------------------------------------------------------------
# Verify Dropout is actually active
# ------------------------------------------------------------

print("\nModel/Dropout State")
print("=" * 50)
print(f"model.training = {model.training}")
print(f"drop1.training = {model.drop1.training}")
print(f"drop2.training = {model.drop2.training}")
print(f"drop3.training = {model.drop3.training}")
print(f"bn1.training = {model.bn1.training}")
print(f"bn2.training = {model.bn2.training}")

# ------------------------------------------------------------
# Variance distribution
# ------------------------------------------------------------

print("\nMC Dropout Variance Distribution")
print("=" * 50)

print(f"50th percentile (median): {np.percentile(mc_variance_corrected, 50):.10f}")
print(f"90th percentile:          {np.percentile(mc_variance_corrected, 90):.10f}")
print(f"95th percentile:          {np.percentile(mc_variance_corrected, 95):.10f}")
print(f"99th percentile:          {np.percentile(mc_variance_corrected, 99):.10f}")
print(f"99.9th percentile:        {np.percentile(mc_variance_corrected, 99.9):.10f}")
print(f"Maximum:                  {np.max(mc_variance_corrected):.10f}")

# ------------------------------------------------------------
# Test ROC-AUC using MC mean probability
# ------------------------------------------------------------

mc_auc_corrected = roc_auc_score(
    y_test_processed,
    mc_mean_corrected
)

print("\nTest ROC-AUC using MC Mean Probability")
print("=" * 50)
print(f"ROC-AUC: {mc_auc_corrected:.4f}")

Corrected MC Dropout Inference
MC passes (T): 50
Prediction shape: torch.Size([50, 38052])
Mean prediction shape: (38052,)
Variance shape: (38052,)

Model/Dropout State
model.training = True
drop1.training = True
drop2.training = True
drop3.training = True
bn1.training = True
bn2.training = True

MC Dropout Variance Distribution
50th percentile (median): 0.0014238660
90th percentile:          0.0028060151
95th percentile:          0.0033530323
99th percentile:          0.0048243385
99.9th percentile:        0.0083678355
Maximum:                  0.0168061405

Test ROC-AUC using MC Mean Probability
ROC-AUC: 0.8293


In [86]:
# ============================================================
# STEP 1D — MC Dropout with BatchNorm Kept in Evaluation Mode
# ============================================================

T = 50

# Start from evaluation mode
# This keeps BatchNorm fixed using the statistics learned during training.
model.eval()

# Activate ONLY Dropout layers
for module in model.modules():
    if isinstance(module, nn.Dropout):
        module.train()

# Verify states before inference
print("MC Dropout Inference State")
print("=" * 50)
print(f"model.training = {model.training}")
print(f"drop1.training = {model.drop1.training}")
print(f"drop2.training = {model.drop2.training}")
print(f"drop3.training = {model.drop3.training}")
print(f"bn1.training = {model.bn1.training}")
print(f"bn2.training = {model.bn2.training}")

# ------------------------------------------------------------
# Run T = 50 stochastic forward passes
# ------------------------------------------------------------

mc_predictions_final = []

with torch.no_grad():
    for _ in range(T):

        batch_predictions = []

        for X_batch, _ in test_loader:
            predictions = model(X_batch)
            batch_predictions.append(
                predictions.squeeze(1).cpu()
            )

        pass_predictions = torch.cat(batch_predictions)
        mc_predictions_final.append(pass_predictions)

# Stack predictions
mc_predictions_final = torch.stack(mc_predictions_final)

# ------------------------------------------------------------
# Calculate MC mean and predictive variance
# ------------------------------------------------------------

mc_mean_final = mc_predictions_final.mean(dim=0).numpy()

mc_variance_final = mc_predictions_final.var(dim=0).numpy()

mc_std_final = np.sqrt(mc_variance_final)

print("\nMC Prediction Results")
print("=" * 50)
print(f"Prediction shape: {mc_predictions_final.shape}")
print(f"MC mean shape:    {mc_mean_final.shape}")
print(f"Variance shape:   {mc_variance_final.shape}")

# ------------------------------------------------------------
# Variance distribution
# ------------------------------------------------------------

print("\nMC Dropout Variance Distribution")
print("=" * 50)

print(
    f"50th percentile (median): "
    f"{np.percentile(mc_variance_final, 50):.10f}"
)

print(
    f"90th percentile:          "
    f"{np.percentile(mc_variance_final, 90):.10f}"
)

print(
    f"95th percentile:          "
    f"{np.percentile(mc_variance_final, 95):.10f}"
)

print(
    f"99th percentile:          "
    f"{np.percentile(mc_variance_final, 99):.10f}"
)

print(
    f"99.9th percentile:        "
    f"{np.percentile(mc_variance_final, 99.9):.10f}"
)

print(
    f"Maximum:                  "
    f"{np.max(mc_variance_final):.10f}"
)

# ------------------------------------------------------------
# Test ROC-AUC using MC mean probability
# ------------------------------------------------------------

mc_auc_final = roc_auc_score(
    y_test_processed,
    mc_mean_final
)

print("\nTest ROC-AUC using MC Mean Probability")
print("=" * 50)
print(f"ROC-AUC: {mc_auc_final:.4f}")

MC Dropout Inference State
model.training = False
drop1.training = True
drop2.training = True
drop3.training = True
bn1.training = False
bn2.training = False

MC Prediction Results
Prediction shape: torch.Size([50, 38052])
MC mean shape:    (38052,)
Variance shape:   (38052,)

MC Dropout Variance Distribution
50th percentile (median): 0.0013986706
90th percentile:          0.0027749955
95th percentile:          0.0033128543
99th percentile:          0.0047697742
99.9th percentile:        0.0086126337
Maximum:                  0.0175067578

Test ROC-AUC using MC Mean Probability
ROC-AUC: 0.8303


In [87]:
# ============================================================
# STEP 1E — Final PRD Threshold Verification
# ============================================================

threshold_sets = {
    "Original PRD (0.05, 0.15)": (0.05, 0.15),
    "Adjusted PRD (0.03, 0.10)": (0.03, 0.10)
}

print("FINAL UNCERTAINTY THRESHOLD VERIFICATION")
print("=" * 65)

for name, (low_threshold, high_threshold) in threshold_sets.items():

    strata = np.where(
        mc_variance_final < low_threshold,
        "LOW",
        np.where(
            mc_variance_final < high_threshold,
            "MEDIUM",
            "HIGH"
        )
    )

    counts = pd.Series(strata).value_counts()
    counts = counts.reindex(
        ["LOW", "MEDIUM", "HIGH"],
        fill_value=0
    )

    print(f"\n{name}")
    print("-" * 65)
    print(f"LOW    (< {low_threshold}):       {counts['LOW']:,}")
    print(f"MEDIUM ({low_threshold}–<{high_threshold}): {counts['MEDIUM']:,}")
    print(f"HIGH   (≥ {high_threshold}):      {counts['HIGH']:,}")

    print(
        f"Minimum ≥150 check: "
        f"LOW={counts['LOW'] >= 150}, "
        f"MEDIUM={counts['MEDIUM'] >= 150}, "
        f"HIGH={counts['HIGH'] >= 150}"
    )

print("\nFinal observed variance range")
print("-" * 65)
print(f"Minimum σ²: {np.min(mc_variance_final):.10f}")
print(f"Maximum σ²: {np.max(mc_variance_final):.10f}")

FINAL UNCERTAINTY THRESHOLD VERIFICATION

Original PRD (0.05, 0.15)
-----------------------------------------------------------------
LOW    (< 0.05):       38,052
MEDIUM (0.05–<0.15): 0
HIGH   (≥ 0.15):      0
Minimum ≥150 check: LOW=True, MEDIUM=False, HIGH=False

Adjusted PRD (0.03, 0.10)
-----------------------------------------------------------------
LOW    (< 0.03):       38,052
MEDIUM (0.03–<0.1): 0
HIGH   (≥ 0.1):      0
Minimum ≥150 check: LOW=True, MEDIUM=False, HIGH=False

Final observed variance range
-----------------------------------------------------------------
Minimum σ²: 0.0000003084
Maximum σ²: 0.0175067578


In [88]:
# ============================================================
# STEP 1F — Variance Distribution Diagnostic
# ============================================================

print("MC DROPOUT VARIANCE DIAGNOSTIC")
print("=" * 60)

# Count how many observations exceed important variance levels
diagnostic_thresholds = [0.001, 0.002, 0.003, 0.005, 0.01, 0.015, 0.02]

for threshold in diagnostic_thresholds:
    count = np.sum(mc_variance_final >= threshold)
    percentage = (count / len(mc_variance_final)) * 100

    print(
        f"σ² ≥ {threshold:<6}: "
        f"{count:>6,} instances "
        f"({percentage:>6.2f}%)"
    )

print("=" * 60)

# Number of unique variance values
print(f"Total variance observations: {len(mc_variance_final):,}")
print(
    f"Instances with σ² > 0.01: "
    f"{np.sum(mc_variance_final > 0.01):,}"
)

print(
    f"Instances with σ² > 0.015: "
    f"{np.sum(mc_variance_final > 0.015):,}"
)

MC DROPOUT VARIANCE DIAGNOSTIC
σ² ≥ 0.001 : 25,834 instances ( 67.89%)
σ² ≥ 0.002 :  9,985 instances ( 26.24%)
σ² ≥ 0.003 :  2,894 instances (  7.61%)
σ² ≥ 0.005 :    313 instances (  0.82%)
σ² ≥ 0.01  :     21 instances (  0.06%)
σ² ≥ 0.015 :      3 instances (  0.01%)
σ² ≥ 0.02  :      0 instances (  0.00%)
Total variance observations: 38,052
Instances with σ² > 0.01: 21
Instances with σ² > 0.015: 3


In [89]:
# ============================================================
# STEP 1 — Final Percentile-Based Uncertainty Stratification
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Compute percentile boundaries
# ------------------------------------------------------------

p33 = np.percentile(mc_variance_final, 33)
p66 = np.percentile(mc_variance_final, 66)

print("Percentile-Based Uncertainty Stratification")
print("=" * 65)
print(f"33rd percentile σ²: {p33:.10f}")
print(f"66th percentile σ²: {p66:.10f}")
print()

# ------------------------------------------------------------
# 2. Rank all instances by predictive variance
# ------------------------------------------------------------

n_instances = len(mc_variance_final)

sorted_indices = np.argsort(mc_variance_final)

# Each stratum contains exactly one-third of the test set
stratum_size = n_instances // 3

low_indices = sorted_indices[:stratum_size]
medium_indices = sorted_indices[stratum_size:2 * stratum_size]
high_indices = sorted_indices[2 * stratum_size:]

# ------------------------------------------------------------
# 3. Create stratum labels
# ------------------------------------------------------------

final_stratum = np.empty(n_instances, dtype=object)

final_stratum[low_indices] = "LOW"
final_stratum[medium_indices] = "MEDIUM"
final_stratum[high_indices] = "HIGH"

# ------------------------------------------------------------
# 4. Create diagnostic DataFrame
# ------------------------------------------------------------

stratification_df = pd.DataFrame({
    "test_instance_id": np.arange(n_instances),
    "sigma_squared": mc_variance_final,
    "uncertainty_stratum": final_stratum
})

# ------------------------------------------------------------
# 5. Report stratum sizes
# ------------------------------------------------------------

stratum_counts = (
    stratification_df["uncertainty_stratum"]
    .value_counts()
    .reindex(["LOW", "MEDIUM", "HIGH"])
)

print("Stratum Sizes")
print("-" * 65)

for stratum in ["LOW", "MEDIUM", "HIGH"]:
    print(f"{stratum:<10}: {stratum_counts[stratum]:,}")

print("-" * 65)
print(f"Total     : {stratum_counts.sum():,}")

# ------------------------------------------------------------
# 6. Report actual σ² range within each stratum
# ------------------------------------------------------------

print("\nσ² Range Within Each Stratum")
print("-" * 65)

for stratum in ["LOW", "MEDIUM", "HIGH"]:

    values = stratification_df.loc[
        stratification_df["uncertainty_stratum"] == stratum,
        "sigma_squared"
    ]

    print(
        f"{stratum:<10}: "
        f"min = {values.min():.10f}, "
        f"max = {values.max():.10f}"
    )

# ------------------------------------------------------------
# 7. Verify the 150-instance minimum requirement
# ------------------------------------------------------------

print("\nMinimum Stratum Requirement")
print("-" * 65)

for stratum in ["LOW", "MEDIUM", "HIGH"]:
    print(
        f"{stratum:<10} ≥ 150: "
        f"{stratum_counts[stratum] >= 150}"
    )

# ------------------------------------------------------------
# 8. Verify exact equal-sized strata
# ------------------------------------------------------------

expected_size = n_instances // 3

print("\nEqual-Stratum Verification")
print("-" * 65)
print(f"Expected per stratum: {expected_size:,}")
print(
    f"All strata exactly equal: "
    f"{all(stratum_counts == expected_size)}"
)

# ------------------------------------------------------------
# 9. Sample 200 instances from each stratum
# ------------------------------------------------------------

RANDOM_STATE = 42
SAMPLE_SIZE = 200

sampled_parts = []

for stratum in ["LOW", "MEDIUM", "HIGH"]:

    stratum_data = stratification_df[
        stratification_df["uncertainty_stratum"] == stratum
    ]

    sampled = stratum_data.sample(
        n=SAMPLE_SIZE,
        random_state=RANDOM_STATE
    )

    sampled_parts.append(sampled)

stratified_samples = (
    pd.concat(sampled_parts)
    .sort_values(
        ["uncertainty_stratum", "test_instance_id"]
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 10. Verify final sample
# ------------------------------------------------------------

print("\nFinal XAI Sample")
print("=" * 65)
print(f"Total sampled instances: {len(stratified_samples):,}")

print("\nSamples per stratum:")

sample_counts = (
    stratified_samples["uncertainty_stratum"]
    .value_counts()
    .reindex(["LOW", "MEDIUM", "HIGH"])
)

for stratum in ["LOW", "MEDIUM", "HIGH"]:
    print(f"{stratum:<10}: {sample_counts[stratum]:,}")

# ------------------------------------------------------------
# 11. Save the 600-instance stratified sample
# ------------------------------------------------------------

output_file = "stratified_samples.csv"

stratified_samples.to_csv(
    output_file,
    index=False
)

print("\nSaved successfully:")
print(output_file)

Percentile-Based Uncertainty Stratification
33rd percentile σ²: 0.0010218532
66th percentile σ²: 0.0017743841

Stratum Sizes
-----------------------------------------------------------------
LOW       : 12,684
MEDIUM    : 12,684
HIGH      : 12,684
-----------------------------------------------------------------
Total     : 38,052

σ² Range Within Each Stratum
-----------------------------------------------------------------
LOW       : min = 0.0000003084, max = 0.0010297309
MEDIUM    : min = 0.0010297557, max = 0.0017923317
HIGH      : min = 0.0017924170, max = 0.0175067578

Minimum Stratum Requirement
-----------------------------------------------------------------
LOW        ≥ 150: True
MEDIUM     ≥ 150: True
HIGH       ≥ 150: True

Equal-Stratum Verification
-----------------------------------------------------------------
Expected per stratum: 12,684
All strata exactly equal: True

Final XAI Sample
Total sampled instances: 600

Samples per stratum:
LOW       : 200
MEDIUM    : 200